In [1]:
# Importações

import pandas as pd
import numpy as np
from collections import defaultdict
import math
import itertools
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_validate
from sklearn.metrics import recall_score, make_scorer
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB

In [2]:
# Leitura dos arquivos extraidos da base HGNC
d1 = pd.read_csv("GENES\\family.csv", sep=",", dtype=str)    # Informações das famílias gênicas
d2 = pd.read_csv("GENES\\gene_has_family.csv", sep=",", dtype=str)   # informações de quais genes pertencem as suas repesctivas famílias
d3 = pd.read_csv("GENES\\genes.tsv", sep="\t", dtype=str)    # Informações de todos os genes catalogados pela HGNC

In [ ]:
df1 = pd.DataFrame()

# Carregar conjunto de dados
def Conjunto():
    # Pedir para o usuário informar o conjunto de dados a ser carregado.
    caminho = input("Digite o número do conjunto de dados a ser carregado: \n\n1- Ulcerative Colitis\n2- Glioma\n3- Metastatic prostate cancer (HG-U95C)\n4- Metastatic prostate cancer (HG-U95A)\n5- Lung cancer\n6- Lung adenocarcinoma\n7- Leukemia\n8- Pulmonary hypertension\n9- Non-small cell lung carcinoma\n10- Colorectal cancer \n\n")

    global df1
    
    if caminho == '1':       # ulcerative colitis
        df = pd.read_csv("Atual\\Colitis.csv", header=None, low_memory=False)

        #Carregar lista de genes associado a este conjunto. (Realizado em todos os conjuntos de dados abaixo).
        df1 = pd.read_csv("GENES\\colitis.tsv", sep="\t")
        
    elif caminho == '2':         # Glioma
        df = pd.read_csv("Atual\\Glioma.csv", header=None, low_memory=False)
        
        #Genes
        df1 = pd.read_csv("GENES\\Glioma.tsv", sep="\t")
    
    elif caminho == '3':      # Metastatic prostate cancer (HG-U95C)
        df = pd.read_csv("Atual\\Prostate1.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\Prostate.tsv", sep="\t")

    elif caminho == '4':       # Metastatic prostate cancer (HG-U95A)
        df = pd.read_csv("Atual\\Prostate2.csv", header=None, low_memory=False)

        # Genes
        df1 = pd.read_csv("GENES\\Prostate.tsv", sep="\t")
        
    elif caminho == '5':   # lung cancer
        df = pd.read_csv("Atual\\Lung1.csv", header=None, low_memory=False)
    
        # Genes
        df1 = pd.read_csv("GENES\\LungCarcinoma.tsv", sep="\t")
        
    elif caminho == '6':       # Lung adenocarcinoma
        df = pd.read_csv("Atual\\Lung2.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\AdenocarcinomaLung.tsv", sep="\t")
        
    elif caminho == '7':       # Leukemia
        df = pd.read_csv("Atual\\Leukemia.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\leukemia.tsv", sep="\t")
    
    elif caminho == '8':      # Pulmonary hypertension
        df = pd.read_csv("Atual\\Pulmonary.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\pulmonary.tsv", sep="\t")
        
    elif caminho == '9':       # Non-small cell lung carcinoma
        df = pd.read_csv("Atual\\nonsmall.csv", header=None, low_memory=False)

        # Genes
        df1 = pd.read_csv("GENES\\non-small.tsv", sep="\t")
        
    elif caminho == '10':     # Colorectal cancer
        df = pd.read_csv("Atual\\Colorectal.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\colorectal.tsv", sep="\t")
        
    else:
        return 0

    # Resolver o problema do pandas identificar atributos iguais e mudar o nome (Conjuntos possuem genes iguais com diferentes valores de expressão)
    novo = df.iloc[0]
    df.columns = novo
    df = df.drop(0).reset_index(drop=True)
    
    print('Conjunto de dados:\n')
    print(df)
    
    return df

In [ ]:
clas = []

# Gerar os subconjuntos com base nas informações da DisGeNet e HGNC
def SubConjuntos():
    # Obtem o conjunto de dados para gerar os subconjuntos
    conjunto = Conjunto()

    # Redução de Dimensionalidade pelos genes da DisgenNET
    arr = []

    # Todos os genes do conjunto de dados que estiverem na lista da DisGeNet serão retidos.
    for i in df1['Gene']:
        if i in conjunto.columns:
            arr.append(i)

    arr.append('Classe')
    # Filtra o conjunto de dados com os genes retidos e a classe.
    fim = conjunto.loc[:, arr]

    print("Conjunto reduzido:\n")
    print(fim)
    
    global clas
    clas = fim['Classe'].astype('int')
    fim = fim.drop('Classe', axis=1)
    
    # Gerar os subconjuntos por Familias

    # Combinar os conjuntos da HGNC pelo ID do gene para obter sua respectiva família
    result = pd.merge(d2, d3, on=['hgnc_id'], how="inner")
    result = result.loc[:, ['family_id', 'symbol']].astype({'family_id': int, 'symbol': str}) 
    
    r = result[result['symbol'].isin(fim.columns)]   # Filtrar os genes que estão apenas no conjunto de dados
    r = r.sort_values('family_id')    # Ordenar a tabela do menor para a maior id da família.
    
    familias = []    #  Lista de dicionários final que irá conter a relação de genes com sua família             
    genes = []                 
    fam_atual = None              

    for _, linha in r.iterrows():      # Para cada linha da tabela
        fid, gene = linha['family_id'], linha['symbol']   # Retem o id da família e o seu gene

        # Uma vez que a tabela já está ordenada, os genes das próximas linhas serão da mesma família
        # Se mudou a família, salva o bloco anterior e zera o acumulador
        if fid != fam_atual and fam_atual is not None:
            familias.append({fam_atual: genes})
            genes = []                 # reinicia a lista para o próximo bloco

        # Continua no mesmo (ou inicia o novo) bloco
        genes.append(gene)
        fam_atual = fid

    # Depois do laço, grava o último bloco
    if genes:
        familias.append({fam_atual: genes})

    # Gera uma lista de subconjuntos passando o id da família para cada "nome" do subconjunto por meio do parâmetro attrs
    listaa = []

    for familia in familias:
        # Pega a primeira chave e o primeiro valor do dicionário
        id_familia = next(iter(familia.keys()))
        colunas_familia = next(iter(familia.values()))
        
        # Faz a seleção das colunas do conjunto de dados
        _df = fim.loc[:, colunas_familia]
        
        # Adiciona o atributo 'Família' nos metadados do conjunto de dados
        _df.attrs["Família"] = id_familia
        
        # Adiciona o subconjunto na lista final
        listaa.append(_df)

    
    # subdividir os subconjuntos grandes pela média geral das colunas dos subconjuntos
    med = 0

    for i in listaa:
        med = med + len(i.columns)

    me = med / len(listaa)
    me = round(me)

    # Após obter a média, os subconjuntos com atributos maiores que ela seram subdividos.
    lista = []

    for i in listaa:
        if len(i.columns) > me:
            # Obter a quantidade de subdivisões
            x = len(i.columns) / me
            count = 0
            for d in range(math.ceil(x)):
                lista.append(i.iloc[:,count:count+me].astype('float'))
                count = count + me
        else:
            lista.append(i.astype('float'))
            
    print('Quantidade de subconjuntos gerados: ', len(lista))
    
    return lista

In [ ]:
# Modelo Final
def Modelo():
    #Obtem a lista de subconjuntos ou o conjunto de dados original
    lista = SubConjuntos()
    
    #Ranquear os subconjuntos
    
    cvr = ShuffleSplit(n_splits=100, test_size=0.25, random_state=42) # divisões para os subconjuntos
    cvm = ShuffleSplit(n_splits=100, test_size=0.25, random_state=41) # divisão para o modelo final
    
    resul = defaultdict(list)
    
    ml = input("Digite o algoritmo: \n\n1- Random Forest\n2 - HistGBM\n3- GB\n4- KNN\n5- SVM\n6- NaiveBayes\n")
    
    if ml == '1':
        model = RandomForestClassifier(n_estimators=50, random_state=42)
      
    elif ml == '2':      
        model = HistGradientBoostingClassifier(max_iter=50, min_samples_leaf=6, random_state=42)

    elif ml == '3':      
        model = GradientBoostingClassifier(n_estimators=50, random_state=42)

    elif ml == '4':
        model = KNeighborsClassifier(n_neighbors=17)

    elif ml == '5':
        model = LinearSVC(max_iter=100000, random_state=42)

    elif ml == '6':
        model = GaussianNB()
        
    # Ranqueia cada subconjunto obtendo a sua acurácia.
    for idx, i in enumerate(lista):
        scores = cross_validate(model, i, clas,scoring='accuracy', cv=cvr, n_jobs=-1)

        resul[idx].append(scores['test_score'].mean())

    # Ordena da maior acurácia para a menor
    ordenados = sorted(resul.items(), key=lambda item: item[1], reverse=True)
    ordenados = dict(ordenados)
    
    # Métricas de desempenho da classificação
    sensit = make_scorer(recall_score, pos_label=1)
    specif = make_scorer(recall_score, pos_label=0)
    scorers = {'accuracy': 'accuracy', 'roc_auc': 'roc_auc', 'f1': 'f1', 'precision': 'precision', 'sensit': sensit, 'specif': specif}

    # Modelo Final
    # Para o conjunto de dados de número 10 foi verificado o menor numero de atributos entre as acurácias de 100%, sendo apenas 1 atributo.
    
    count = 1 # Top 10
    iguais = []
    dados = []
    
    for chave, valor in ordenados.items():
        if count < 11:
            # imprime os genes do melhor subconjunto ranqueado e a sua família pelos metadadados .attrs
            print(count,"° do ranking: ", valor[0], lista[chave].attrs, 'Genes: ',lista[chave].columns.to_list(),'\n')
            
            if count == 1:
                # Avaliar modelo com todas as métricas
                scores = cross_validate(model, lista[chave], clas, cv=cvr, scoring=scorers, n_jobs=-1)
                m_auc = scores['test_roc_auc'].mean()
                m_f1 = scores['test_f1'].mean()
                m_prec = scores['test_precision'].mean()
                m_sensit = scores['test_sensit'].mean()
                m_specif = scores['test_specif'].mean()
                print('1° Grupo - ROC_AUC: ', m_auc, '\nF1: ', m_f1,'\nPrecision: ', m_prec, '\nSensitivity: ', m_sensit, '\nSpecificity: ', m_specif,'\n')
                menor = valor[0]

            dados.append(lista[chave])
            count = count + 1

    for num in range(2, 11):
        # Realiza todas as combinações possiveis dos 10 primeiros subconjuntos
        combina = list(itertools.combinations(dados, num))
        maior = menor
        anterior = dados[0]

        for sub in combina:
            # une cada combinação em um subconjunto
            conjunto = pd.concat(sub, axis=1)

            # Verificar se há duplicados para identificar genes possivelmente relevantes para a doença em análise
            for i in range(0,len(conjunto.columns)):
                for j in range(0,len(conjunto.columns)):
                    if i != j:
                        if (conjunto.iloc[:, i] == conjunto.iloc[:, j]).all():
                            iguais.append(conjunto.columns[i])
                            
            # Se tiver duplicatas irá remover.
            conjunto = conjunto.T.drop_duplicates(keep='first').T

            # Avaliar modelo com todas as métricas
            scor = cross_validate(model, conjunto, clas, cv=cvm, scoring=scorers, n_jobs=-1)
            
            # Identifica qual subconjunto obteve a melhor acurácia
            media = scor['test_accuracy'].mean()

            if media > maior:
                anterior = conjunto
                maior = media
                m_auc = scor['test_roc_auc'].mean()
                m_f1 = scor['test_f1'].mean()
                m_prec = scor['test_precision'].mean()
                m_sensit = scor['test_sensit'].mean()
                m_specif = scor['test_specif'].mean()

        # imprime todas as métricas para o mesmo subconjunto        
        print(num,"° Grupo -- Genes: ", anterior.columns.to_list())
        print('ACC: ', maior ,'\nAUC: ', m_auc, '\nF1: ', m_f1,'\nPrecision: ', m_prec, '\nSensitivity: ', m_sensit, '\nSpecificity: ', m_specif,'\n')
        
    print('Genes iguais: ', set(iguais))

In [ ]:
# Uma vez que é mostrado para o usuário apenas o id da familía, aqui mostrará o nome da família correspondente.
def Converter():
    fami = input("Digite o ID da família ou 0 para sair: \n")
    
    if fami == '0':
        exit()
        
    else:    
        identif = d1.loc[d1['id'] == fami]
        identif = identif.iat[0,2]
    
        print('Nome da Família: ', identif)
        
        Converter()

In [ ]:
# Execução principal
if __name__ == "__main__":
    
    digi = input('Digite o número correspondente: \n\n1 - Carregar conjuntos\n2- Converter Famílias\n')
    
    if digi == '1':
        Modelo()
        
    elif digi == '2':
        Converter()
        
    else:
        exit()